1. 5분위 NAV, 로그수익률
2. 히트맵 샤프
3. 롱-온리 NAV, 로그 수익률
4. 롱-숏 NAV, 로그 수익률
5. 거래비용..

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# 경로 설정
BASE = Path(r"c:\Users\XH58\Desktop\Quantifi\02_2026-1\02_메인 세션\02_전략 탐색\15_JMQ\01_output\01_quarterly\01_long_term")

FILES = {
    "quintile": BASE / "5분위_portfolio_return_all.csv",
    "double_sort_25": BASE / "double_sort_portfolio_cap_return_25.csv",
    "spread": BASE / "spread_portfolio_return.csv",
}

OUT_DIR = BASE  # 같은 폴더에 저장; 바꾸려면 수정


def enrich_nav_log_cum(df: pd.DataFrame, date_col: str = "Date") -> pd.DataFrame:
    """수익률 열에 대해 NAV, 로그 누적수익률 열을 추가한 wide DataFrame 반환."""
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    out = out.sort_values(date_col).reset_index(drop=True)

    ret_cols = [c for c in out.columns if c != date_col]

    for c in ret_cols:
        r = pd.to_numeric(out[c], errors="coerce")
        # 1+r <= 0 이면 log 불가 — 해당 구간은 NaN 처리
        gross = 1.0 + r
        nav = gross.cumprod()
        log_cum = np.where(gross > 0, np.log(gross).cumsum(), np.nan)

        out[f"NAV_{c}"] = nav
        out[f"log_cum_{c}"] = log_cum

    return out


def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    for name, path in FILES.items():
        df = pd.read_csv(path)
        enriched = enrich_nav_log_cum(df)
        out_path = OUT_DIR / f"{path.stem}_nav_log_cum.csv"
        enriched.to_csv(out_path, index=False)
        print(f"saved: {out_path}")


if __name__ == "__main__":
    main()

saved: c:\Users\XH58\Desktop\Quantifi\02_2026-1\02_메인 세션\02_전략 탐색\15_JMQ\01_output\01_quarterly\01_long_term\5분위_portfolio_return_all_nav_log_cum.csv
saved: c:\Users\XH58\Desktop\Quantifi\02_2026-1\02_메인 세션\02_전략 탐색\15_JMQ\01_output\01_quarterly\01_long_term\double_sort_portfolio_cap_return_25_nav_log_cum.csv
saved: c:\Users\XH58\Desktop\Quantifi\02_2026-1\02_메인 세션\02_전략 탐색\15_JMQ\01_output\01_quarterly\01_long_term\spread_portfolio_return_nav_log_cum.csv


In [16]:
import numpy as np
import pandas as pd
import plotly.express as px

# 경로는 노트북 위치에 맞게 조정
PATH = "./01_output/01_quarterly/01_long_term/double_sort_portfolio_cap_return_25.csv"

PERIODS_PER_YEAR = 4  # module_quarter.performance_metrics 와 동일 (분기)


def sharpe_ratio_quarterly(series: pd.Series) -> float:
    """module_quarter.performance_metrics 와 동일 정의 (Rf=0, 분기→연율화)."""
    r = pd.to_numeric(series, errors="coerce").dropna()
    if r.empty:
        return np.nan
    mean_q = r.mean()
    vol_q = r.std()
    vol_ann = vol_q * np.sqrt(PERIODS_PER_YEAR)
    if vol_ann == 0:
        return np.nan
    return (mean_q * PERIODS_PER_YEAR) / vol_ann


# 1) 수익률 로드 (Date 인덱스)
df = pd.read_csv(PATH, parse_dates=["Date"]).set_index("Date").sort_index()

# 2) 열별 샤프 → MultiIndex → 5×5 (노트북 CAGR 히트맵과 동일 패턴)
sharpes = df.apply(sharpe_ratio_quarterly)
sharpes.index = sharpes.index.str.split("_", expand=True)
sharpes.index.names = ["C", "Q"]
sharpe_matrix = sharpes.unstack()  # 행: C, 열: Q

# 3) 시각화 (CAGR 셀과 동일하게 px.imshow)
fig = px.imshow(
    sharpe_matrix,
    text_auto=".2f",
    color_continuous_scale="YlOrRd",
    aspect="equal",
    title="Sharpe Ratio Heatmap by Cap vs Factor Quintile (Quarterly, Rf=0)",
    width=600,
    height=600,
    labels=dict(x="Factor Quintile (Q)", y="Market Cap Quintile (C)", color="Sharpe"),
)
fig.update_yaxes(autorange="reversed")  # 노트북 CAGR 히트맵 y축과 동일
fig.show()

In [17]:
import numpy as np
import pandas as pd
import plotly.express as px

# 경로는 노트북 위치에 맞게 조정
PATH = "01_output/01_quarterly/01_long_term/double_sort_portfolio_equal_return_25.csv"

PERIODS_PER_YEAR = 4  # module_quarter.performance_metrics 와 동일 (분기)


def sharpe_ratio_quarterly(series: pd.Series) -> float:
    """module_quarter.performance_metrics 와 동일 정의 (Rf=0, 분기→연율화)."""
    r = pd.to_numeric(series, errors="coerce").dropna()
    if r.empty:
        return np.nan
    mean_q = r.mean()
    vol_q = r.std()
    vol_ann = vol_q * np.sqrt(PERIODS_PER_YEAR)
    if vol_ann == 0:
        return np.nan
    return (mean_q * PERIODS_PER_YEAR) / vol_ann


# 1) 수익률 로드 (Date 인덱스)
df = pd.read_csv(PATH, parse_dates=["Date"]).set_index("Date").sort_index()

# 2) 열별 샤프 → MultiIndex → 5×5 (노트북 CAGR 히트맵과 동일 패턴)
sharpes = df.apply(sharpe_ratio_quarterly)
sharpes.index = sharpes.index.str.split("_", expand=True)
sharpes.index.names = ["C", "Q"]
sharpe_matrix = sharpes.unstack()  # 행: C, 열: Q

# 3) 시각화 (CAGR 셀과 동일하게 px.imshow)
fig = px.imshow(
    sharpe_matrix,
    text_auto=".2f",
    color_continuous_scale="YlOrRd",
    aspect="equal",
    title="Sharpe Ratio Heatmap by Cap vs Factor Quintile (Quarterly, Rf=0)",
    width=600,
    height=600,
    labels=dict(x="Factor Quintile (Q)", y="Market Cap Quintile (C)", color="Sharpe"),
)
fig.update_yaxes(autorange="reversed")  # 노트북 CAGR 히트맵 y축과 동일
fig.show()